In [29]:
# Setup and imports

import sys
from pathlib import Path
import importlib
import inspect

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.data_source.wikipedia import WikipediaSource


import src.pipeline.indexing_rag as indexing_module
import src.pipeline.retrieval_rag as retrieval_module
import src.pipeline.generation_rag as generation_module


from src.query_translation.multi_query import (
    create_multi_query_generator,
    create_multi_query_retrieval_chain,
)

from src.advanced_indexing.raptor import RaptorIndexer


from src.Reranking.reranking import CrossEncoderReranker


from src.advanced_RAG.Self_RAG.self_rag import SelfRAG
from src.advanced_RAG.Long_Context.long_context import LongContext


from src.memory.memory import *


from src.evaluation.rag_evaluation import RAGEvaluator

from data_source.wikipedia import WikipediaSource
from data_source.web_search import WebSearchSource

importlib.reload(indexing_module)
importlib.reload(retrieval_module)
importlib.reload(generation_module)


build_vectorstore = indexing_module.build_vectorstore
split_documents = indexing_module.split_documents

create_retriever = retrieval_module.create_retriever
retrieve_documents = retrieval_module.retrieve_documents


print("Imports successful.")

Imports successful.


In [7]:
# Local models

llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
    base_url="http://127.0.0.1:11434",
)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://127.0.0.1:11434",
)

print("Models initialized.")

Models initialized.


In [8]:
# Test embeddings

test_embedding = embeddings.embed_query("test")

print("Embedding dimension:", len(test_embedding))

Embedding dimension: 768


In [9]:
# User query

query = input("Ask a question: ")

print("\nQuestion:", query)


Question: today nepali date 


In [10]:
# Data sources - Wikipedia + Web Search

wikipedia = WikipediaSource(
    top_k=5,
)

web_search = WebSearchSource(
    top_k=5,
)

wikipedia_documents = wikipedia.retrieve(query)

web_documents = web_search.retrieve(query)

documents = wikipedia_documents + web_documents

print("Wikipedia documents:", len(wikipedia_documents))
print("Web search documents:", len(web_documents))
print("Total documents:", len(documents))

Wikipedia documents: 5
Web search documents: 4
Total documents: 9


In [11]:
# Validate combined documents

for i, document in enumerate(documents, 1):
    print(f"\nDocument {i}")
    print("Source:", document.metadata.get("source"))
    print("Title:", document.metadata.get("title"))
    print("URL:", document.metadata.get("url"))
    print("Content length:", len(document.page_content))


Document 1
Source: wikipedia
Title: Nepali Army
URL: https://en.wikipedia.org/wiki/Nepali_Army
Content length: 55724

Document 2
Source: wikipedia
Title: Nirmal Purja
URL: https://en.wikipedia.org/wiki/Nirmal_Purja
Content length: 64266

Document 3
Source: wikipedia
Title: Nepalese passport
URL: https://en.wikipedia.org/wiki/Nepalese_passport
Content length: 14567

Document 4
Source: wikipedia
Title: Nepalese Muslims
URL: https://en.wikipedia.org/wiki/Nepalese_Muslims
Content length: 26603

Document 5
Source: wikipedia
Title: 2025 Nepalese Gen Z protests
URL: https://en.wikipedia.org/wiki/2025_Nepalese_Gen_Z_protests
Content length: 98324

Document 6
Source: web_search
Title: Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today
URL: https://nepalipatro.com.np
Content length: 116

Document 7
Source: web_search
Title: Ramro Patro - Nepali Calendar date and time Today
URL: https://www.facebook.com/ramropatro/posts/nepali-calendar-date-and-time-today/1444542164136235
Content length:

In [12]:
# Check document chunks

splits = split_documents(
    documents,
    chunk_size=256,
    chunk_overlap=50,
)

print("Original documents:", len(documents))
print("Generated chunks:", len(splits))

Original documents: 9
Generated chunks: 427


In [13]:
# Test chunk embeddings

test_chunks = [document.page_content for document in splits[:20]]

test_vectors = embeddings.embed_documents(test_chunks)

print("Chunks:", len(test_chunks))
print("Vectors:", len(test_vectors))
print("Vector dimension:", len(test_vectors[0]))

Chunks: 20
Vectors: 20
Vector dimension: 768


In [14]:
# Reload indexing module

importlib.reload(indexing_module)

build_vectorstore = indexing_module.build_vectorstore

print(inspect.signature(build_vectorstore))

(documents: List[langchain_core.documents.base.Document], embedding_model: Union[str, Any] = 'BAAI/bge-small-en-v1.5', chunk_size: int = 256, chunk_overlap: int = 50, collection_name: Optional[str] = None, batch_size: int = 32) -> langchain_community.vectorstores.chroma.Chroma


In [15]:
# Verify indexing implementation

source = inspect.getsource(build_vectorstore)

print(source)

def build_vectorstore(
    documents: List[Document],
    embedding_model: Union[str, Any] = "BAAI/bge-small-en-v1.5",
    chunk_size: int = 256,
    chunk_overlap: int = 50,
    collection_name: Optional[str] = None,
    batch_size: int = 32,
) -> Chroma:
    """Split documents and index them into Chroma in batches."""

    splits = split_documents(
        documents,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    if isinstance(embedding_model, str):
        embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model
        )
    else:
        embeddings = embedding_model

    vectorstore = Chroma(
        collection_name=(
            collection_name or "rag_collection"
        ),
        embedding_function=embeddings,
    )

    for start in range(
        0,
        len(splits),
        batch_size,
    ):
        batch = splits[
            start:start + batch_size
        ]

        vectorstore.add_documents(batch)

    return vector

In [16]:
# Index documents

vectorstore = build_vectorstore(
    documents=documents,
    embedding_model=embeddings,
    batch_size=32,
)

print("Combined documents indexed successfully.")

c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch\src\pipeline\indexing_rag.py:77: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Combined documents indexed successfully.


In [17]:
# Create retriever

retriever = create_retriever(
    vectorstore=vectorstore,
    k=5,
)

print("Retriever created.")

Retriever created.


In [18]:
# Query processing - Multi-Query

multi_query_generator = create_multi_query_generator(
    llm=llm,
)

multi_queries = multi_query_generator.invoke(query)

print("Original Query:")
print(query)

print("\nGenerated Queries:")

for i, generated_query in enumerate(
    multi_queries,
    1,
):
    print(f"{i}. {generated_query}")

Original Query:
today nepali date 

Generated Queries:
1. Here are five alternative versions of the original question to help retrieve relevant documents from a vector database:
2. Today's Nepali date
3. What is the current date in Nepali calendar?
4. Nepali date for today
5. What is the date in Nepali calendar for the current day?
6. Current date in Nepali format
7. What is the current date in the format used in Nepal?
8. Date in Nepali calendar today
9. What is the date in the Nepali calendar for the current day?
10. These alternative questions can help retrieve relevant documents that may not be directly matched by the original question, but still contain the desired information.


In [19]:
# Advanced indexing - RAPTOR

raptor = RaptorIndexer(
    llm=llm,
    embeddings=embeddings,
    n_clusters=3,
)

raptor.build_tree(documents=documents)

raptor_results = raptor.retrieve(
    query=query,
    k=3,
)

print("RAPTOR leaf results:", len(raptor_results["leaf"]))
print("RAPTOR cluster results:", len(raptor_results["clusters"]))

RAPTOR leaf results: 3
RAPTOR cluster results: 3


In [20]:
# Multi-Query retrieval

multi_query_retriever = create_multi_query_retrieval_chain(
    retriever=retriever,
    llm=llm,
)

multi_query_documents = multi_query_retriever.invoke(query)

print(f"Unique documents retrieved: " f"{len(multi_query_documents)}")

Unique documents retrieved: 30


c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch\src\query_translation\multi_query.py:40: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch\src\query_translation\multi_query.py:40: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


In [21]:
# Reranking

reranker = CrossEncoderReranker(
    top_k=5,
)

reranked_documents = reranker.rerank_documents(
    query=query,
    documents=multi_query_documents,
)

print(f"Reranked documents: " f"{len(reranked_documents)}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3620.39it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranked documents: 5


In [22]:
# Reranking scores

ranked_documents = reranker.rerank(
    query=query,
    documents=multi_query_documents,
)

for i, (document, score) in enumerate(
    ranked_documents,
    1,
):
    print(f"\nRank {i}")
    print(f"Score: {score}")
    print(document.page_content[:300])


Rank 1
Score: 7.469637870788574
Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद

Rank 2
Score: 7.469637870788574
Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद

Rank 3
Score: 7.469637870788574
Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद

Rank 4
Score: 7.469637870788574
Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद

Rank 5
Score: 7.469635009765625
Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद


In [23]:
# Advanced RAG - Self-RAG

self_rag = SelfRAG(
    llm=llm,
    retriever=retriever,
)

self_rag_answer = self_rag.invoke(
    query,
    max_retries=2,
)

print("Self-RAG answer:")
print(self_rag_answer)


========== SELF-RAG ==========
Retrieval required: NO
Self-RAG answer:
I can help with that!

To find the Nepali date for today, I would need more information about the current date. Could you please provide me with the current date in the Gregorian calendar (the most widely used calendar in the world)?


In [24]:
# Build context

context = "\n\n".join(document.page_content for document in reranked_documents)

print("Context length:", len(context))
print("\nContext preview:")
print(context[:1000])

Context length: 588

Context preview:
Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद

Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद

Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद

Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद

Nepali Calendar 2083 | Aaja Kati Gate? | Nepali date today · ताजा समाचार · ट्रेण्डिङ · स्वास्थ्य · मनोरञ्जन · खेलकूद


In [25]:
# Generation

generation_prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

If the answer is not available in the context,
say that the information is not available.

Answer clearly and concisely.
"""

response = llm.invoke(generation_prompt)

answer = response.content

print("Answer:")
print(answer)

Answer:
Today Nepali date: Nepali Calendar 2083


In [26]:
# Store conversation memory

conversation_memory = ConversationMemory()

conversation_memory.add_message(HumanMessage(content=query))

conversation_memory.add_message(AIMessage(content=response.content))

In [ ]:
# End-to-end RAG pipeline


def run_rag(query):
    # Multi-Query retrieval

    multi_query_documents = multi_query_retriever.invoke(query)

    if not multi_query_documents:
        return "I could not find relevant information."

    # Reranking

    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=multi_query_documents,
    )

    # Build context

    context = "\n\n".join(document.page_content for document in reranked_documents)

    # Generation

    prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

If the answer is not available in the context,
say that the information is not available.

Answer clearly and concisely.
"""

    response = llm.invoke(prompt)

    # Store conversation memory

    conversation_memory.add_message(HumanMessage(content=query))

    conversation_memory.add_message(AIMessage(content=response.content))
    return response.content

In [28]:
# Run integrated RAG

answer = run_rag(query)
print("\nFinal Answer:")
print(answer)

c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch\src\query_translation\multi_query.py:40: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]



Final Answer:
Today Nepali date: Nepali Calendar 2083
